In [1]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "12345678"
auth=(username, password)
driver = GraphDatabase.driver(uri, auth=(username, password))

In [2]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Neo4jMapReduce") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
def Map(driver):

    query = """
    MATCH (c)
    WHERE c.kind = "Compound"
    OPTIONAL MATCH (c)-[r]->()
    RETURN c.name AS Compound, r.metaedge AS metaedge
    """
    with driver.session() as session:
        result = [dict(record) for record in session.run(query)]

    rdd = spark.sparkContext.parallelize(result)
    pairs = rdd.map(lambda x: (x['Compound'], x['metaedge']))
    return pairs

In [4]:
def Sort(pairs):

    gene_types = ['CbG', 'CuG', 'CdG']
    genes = pairs.filter(lambda x: x[1] in gene_types).map(lambda x: (x[0], 1))

    return genes

In [5]:
def Reduce(genes):
    gene_counts = genes.reduceByKey(lambda a, b: a + b)
    desc = gene_counts.sortBy(lambda x: x[1], ascending=False)
    return desc

In [6]:
def MapReduce(driver):
    result = Map(driver)
    sorted = Sort(result)
    reduced = Reduce(sorted)

    return reduced.take(5)

In [7]:
result = MapReduce(driver)
result

[('Crizotinib', 585),
 ('Dasatinib', 564),
 ('Doxorubicin', 532),
 ('Vinblastine', 523),
 ('Digoxin', 522)]

In [8]:
for compound, gene in result:
    print(f"Compound Name: {compound}, Gene Count: {gene}")

Compound Name: Crizotinib, Gene Count: 585
Compound Name: Dasatinib, Gene Count: 564
Compound Name: Doxorubicin, Gene Count: 532
Compound Name: Vinblastine, Gene Count: 523
Compound Name: Digoxin, Gene Count: 522


In [9]:
spark.stop()
driver.close()